## DDL Gold: pf.gold.dim_libreria  (SCD Type 1)


In [0]:
%sql

DROP TABLE IF EXISTS pf.gold.dim_libreria;

CREATE TABLE IF NOT EXISTS pf.gold.dim_libreria (
    libreria_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'PK - Surrogate Key',
    library_name STRING NOT NULL COMMENT 'BK - nombre de la libreria',
    descripcion STRING COMMENT 'Descripcion',
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT 'UTC',
    PRIMARY KEY (libreria_id),
    CONSTRAINT uniq_dim_libreria UNIQUE (library_name)
)
USING DELTA
TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults = 'supported'
)
COMMENT 'Dimension Libreria - SCD Type 1';

In [0]:
%sql

-- Carga/actualizacion SCD1
MERGE INTO pf.gold.dim_libreria AS t
USING (
    SELECT library_name AS library_name,
           CASE library_name
               WHEN 'transformers' THEN 'Transformers (HF)'
               WHEN 'sentence-transformers' THEN 'Sentence Transformers'
               WHEN 'vllm' THEN 'vLLM / tensor-rt-llm'
               ELSE library_name
           END AS descripcion,
           COUNT(DISTINCT model_id) AS n_models
    FROM pf.silver.modelos
    WHERE library_name IS NOT NULL
    GROUP BY library_name
) AS s
ON t.library_name = s.library_name
WHEN MATCHED THEN
    UPDATE SET t.descripcion = s.descripcion
WHEN NOT MATCHED THEN
    INSERT (library_name, descripcion, _createdAt)
    VALUES (UPPER(s.library_name), s.descripcion, CURRENT_TIMESTAMP());

In [0]:
%sql

SELECT libreria_id, library_name, descripcion FROM pf.gold.dim_libreria ORDER BY library_name;